In [20]:
import random
from rl_policy_lab.grpo_toy import encode, decode, prompt_for, random_valid_expr
prompt = "6/2=3"

encoded = encode(prompt)
decoded = decode(encoded)
prompt == decoded # True

pid = prompt_for([1, 2, 4, 5], 10) # '1 2 4 5:10='
eid = random_valid_expr([1, 2, 4, 5], random.Random()) # int+int

pid+eid

'1 2 4 5:10=(5+2)'

In [21]:
import random
from rl_policy_lab.countdown import generate_puzzle, _uses_allowed_numbers, safe_eval, reward

rand = random.Random()
generate_puzzle(rand) # (int[4], sum)

expr = "abc 1.2.3 4b5.6 7 8 9"
numbers = [1,2,3,4,5,6,7,8,9]
_uses_allowed_numbers(expr, numbers) # True


safe_eval("123+--123") # 246.0


expr = "1+2-3*4/5+6-7*8/9"
target=0.3777778
reward(expr, numbers, target) # 0.1
reward(expr, numbers, target+1) # 0.3
reward(expr+ ".", numbers, target) # 0.0 
reward(expr, numbers[:-1], target) # 0.1


0.1

In [46]:
from rl_policy_lab.grpo_toy import TinyGPT

net = TinyGPT()
p_count = sum(p.numel() for p in net.parameters() if p.requires_grad)
print("number of parameters: ", p_count)
net

number of parameters:  418816


TinyGPT(
  (tok): Embedding(22, 128)
  (pos): Embedding(128, 128)
  (blocks): ModuleList(
    (0-1): 2 x Block(
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
      )
      (ln1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      (ln2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      (mlp): Sequential(
        (0): Linear(in_features=128, out_features=512, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=512, out_features=128, bias=True)
      )
    )
  )
  (ln): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
  (head): Linear(in_features=128, out_features=22, bias=False)
)

In [23]:
from rl_policy_lab.grpo_toy import pretrain_format

pretrain_format(net, rand)

sft step   0 loss 3.3353
sft step  50 loss 1.6456
sft step 100 loss 1.0253
sft step 150 loss 0.7244
sft step 200 loss 0.5960
sft step 250 loss 0.5863
sft step 300 loss 0.5741
sft step 350 loss 0.5613
sft step 399 loss 0.4938
sft done, final loss 0.4938


In [24]:
import torch
#pretrain inference
with torch.no_grad():
    for _ in range(3):
        numbers, target = generate_puzzle(rand, 2)
        ids = encode(prompt_for(numbers, target))
        for _ in range(24):
            inp = torch.tensor([ids[-64:]])
            nxt = int(net(inp)[0, -1].argmax())
            if nxt == 1: # EOS = 1
                break
            ids.append(nxt)
        print(f"{numbers} -> {target} | {decode(ids[len(encode(prompt_for(numbers, target))) :])}")

[14, 6] -> 20 | 14*6
[17, 2] -> 19 | (17+2)
[20, 12] -> 8 | 20+2


## K3: estimate KL by sampling tokens

This standalone example needs no trained model. Imagine one fixed prefix with only two possible next tokens, A and B.

- Current model `p`: A = 0.8, B = 0.2. **We sample from this model.**
- Reference model `q`: A = 0.4, B = 0.6.

Exact KL sums over every token: `sum(p * log(p / q))`.
K3 instead scores a sampled token with `q/p - log(q/p) - 1`, then averages those scores.
The added term `q/p - 1` has expected value zero under `p`, so the expected K3 score equals KL.

In the training code, `new_logp` corresponds to `log(p)` and `ref_logp` to `log(q)` for the same sampled token. Here we sample directly from `p` to demonstrate the estimator's exact expectation; stored rollouts from an older policy only approximate that condition after updates.


In [25]:
import torch

# float64 keeps the numerical comparison precise.
p = torch.tensor([0.8, 0.2], dtype=torch.float64)  # current: A, B
q = torch.tensor([0.4, 0.6], dtype=torch.float64)  # reference: A, B

new_logp = p.log()
ref_logp = q.log()
d = ref_logp - new_logp
k3_per_token = torch.exp(d) - d - 1  # the formula from grpo_toy.py
exact_kl = (p * (new_logp - ref_logp)).sum()

print("Token   p(current)   q(reference)   q/p     K3")
for i, token in enumerate(["A", "B"]):
    print(f"{token:5}   {p[i]:.1f}          {q[i]:.1f}            {q[i]/p[i]:.1f}     {k3_per_token[i]:.6f}")

print(f"\nExact KL:                   {exact_kl:.6f}")
print(f"Expected K3 (weighted by p): {(p * k3_per_token).sum():.6f}")
print(f"Expected added term:         {(p * (q / p - 1)).sum():.6f}")
assert torch.allclose((p * k3_per_token).sum(), exact_kl)


Token   p(current)   q(reference)   q/p     K3
A       0.8          0.4            0.5     0.193147
B       0.2          0.6            3.0     0.901388

Exact KL:                   0.334795
Expected K3 (weighted by p): 0.334795
Expected added term:         -0.000000


### Trying to understand GRPO

In [39]:
def completion_logps(net: TinyGPT, seqs: list[list[int]], plens: list[int]):
    """Per-token logprobs of the completion part of each sequence. Returns padded tensor + mask."""
    PAD=2
    device = next(net.parameters()).device
    maxlen = max(len(s) for s in seqs)
    idx = torch.full((len(seqs), maxlen), PAD, device=device, dtype=torch.long)
    for i, s in enumerate(seqs):
        idx[i, : len(s)] = torch.tensor(s, device=device)
    logp = torch.nn.functional.log_softmax(net(idx)[:, :-1], dim=-1).gather(-1, idx[:, 1:, None]).squeeze(-1)
    pos = torch.arange(idx.shape[1] - 1, device=device)[None]
    plens_t = torch.tensor(plens, device=device)[:, None]
    lens_t = torch.tensor([len(s) for s in seqs], device=device)[:, None]
    mask = (pos >= plens_t - 1) & (pos < lens_t - 1)  # token t predicted from prefix <: completion
    return logp, mask.float()


def grpo_step(net, old_net, ref_net, opt, batch, eps=0.2, beta=0.02, epochs=2):
    seqs, plens, rewards = batch
    with torch.no_grad():
        old_logp, mask = completion_logps(old_net, seqs, plens)
        ref_logp, _ = completion_logps(ref_net, seqs, plens)
    n_tok = mask.sum().clamp_min(1) # DAPO style
    for _ in range(epochs):
        new_logp, _ = completion_logps(net, seqs, plens)
        ratio = torch.exp(new_logp - old_logp)
        adv = rewards[:, None] * mask
        surr = torch.min(ratio * adv, ratio.clamp(1 - eps, 1 + eps) * adv)
        kl = torch.exp(ref_logp - new_logp) - (ref_logp - new_logp) - 1  # k3 estimator
        loss = -(surr * mask).sum() / n_tok + beta * (kl * mask).sum() / n_tok
        opt.zero_grad()
        loss.backward()
        opt.step()
    return float(loss.detach())

In [28]:
import copy
ref_net = copy.deepcopy(net).eval()
for p in ref_net.parameters():
    p.requires_grad_(False)
opt = torch.optim.AdamW(net.parameters(), lr=3e-4)


In [40]:
from rl_policy_lab.grpo_toy import encode, decode, prompt_for, sample
from rl_policy_lab.countdown import reward
group = 12 # number of answer combination of a puzzle.
for it in range(1, 301):
    old_net = copy.deepcopy(net).eval()
    puzzles = [generate_puzzle(rand, 2) for _ in range(16)]
    seqs, plens, rws = [], [], []
    for numbers, target in puzzles:
        pids = encode(prompt_for(numbers, target))
        rollout = sample(old_net, pids, group)
        for s in rollout:
            seqs.append(s)
            plens.append(len(pids))
            rws.append(reward(decode(s[len(pids) :]), numbers, target))
    # group-normalized advantages (the heart of GRPO: no value network)
    rws_t = torch.tensor(rws)
    adv = torch.zeros_like(rws_t)
    for i in range(16):
        g = rws_t[i * group : (i + 1) * group]
        # disadvantage: all result in a group may be the same.
        adv[i * group : (i + 1) * group] = (g - g.mean()) / (g.std() + 1e-4)
    loss = grpo_step(net, old_net, ref_net, opt, (seqs, plens, adv))
    if it % 25 == 0 or it == 1:
        exact = sum(1 for r in rws if r >= 1.0) / len(rws)
        print(f"iter {it:3d} loss {loss:+.4f} mean_r {rws_t.mean():.3f} exact {exact:.2f}", flush=True)

iter   1 loss -0.0129 mean_r 0.426 exact 0.27
iter  25 loss -0.0233 mean_r 0.442 exact 0.28
iter  50 loss -0.0098 mean_r 0.443 exact 0.27
iter  75 loss +0.0171 mean_r 0.626 exact 0.48
iter 100 loss -0.0106 mean_r 0.584 exact 0.44
iter 125 loss -0.0283 mean_r 0.662 exact 0.53
iter 150 loss -0.0011 mean_r 0.699 exact 0.58
iter 175 loss -0.0263 mean_r 0.613 exact 0.49
iter 200 loss +0.0160 mean_r 0.627 exact 0.48
iter 225 loss +0.0149 mean_r 0.580 exact 0.41
iter 250 loss -0.0140 mean_r 0.599 exact 0.44
iter 275 loss +0.0116 mean_r 0.638 exact 0.48
iter 300 loss -0.0129 mean_r 0.744 exact 0.64


In [41]:
net.eval()
with torch.no_grad():
    for _ in range(3):
        numbers, target = generate_puzzle(rng, n_numbers)
        ids = encode(prompt_for(numbers, target))
        for _ in range(24):
            inp = torch.tensor([ids[-64:]], device=device)
            nxt = int(net(inp)[0, -1].argmax())
            if nxt == 1:
                break
            ids.append(nxt)
        print(f"{numbers} -> {target} | {decode(ids[len(encode(prompt_for(numbers, target))) :])}")

iter   1 loss -0.0078 mean_r 0.605 exact 0.44
iter  25 loss -0.0116 mean_r 0.370 exact 0.29
iter  50 loss -0.0105 mean_r 0.318 exact 0.19
iter  75 loss -0.0134 mean_r 0.424 exact 0.32
iter 100 loss -0.0213 mean_r 0.417 exact 0.30


In [ ]:
from rl_policy_lab.grpo_toy import train
metrics = train()

sft step   0 loss 3.4414
sft step  50 loss 1.7071
sft step 100 loss 1.2445
sft step 150 loss 0.7024
sft step 200 loss 0.5773
sft step 250 loss 0.5987
sft step 300 loss 0.5866
sft step 350 loss 0.5782
sft step 399 loss 0.5675
sft done, final loss 0.5675
iter   1 loss -0.0095 mean_r 0.355 exact 0.17
iter  25 loss -0.0088 mean_r 0.477 exact 0.32
iter  50 loss -0.0100 mean_r 0.620 exact 0.47
iter  75 loss +0.0116 mean_r 0.580 exact 0.42
iter 100 loss +0.0051 mean_r 0.621 exact 0.48
iter 125 loss -0.0040 mean_r 0.769 exact 0.68
iter 150 loss -0.0020 mean_r 0.757 exact 0.66
iter 175 loss +0.0058 mean_r 0.799 exact 0.73


In [ ]:
from rl_policy_lab import show_metrics

show_metrics(metrics)
